# FM Fundamentals

This notebook pulls together the core FM material from the legacy notebooks: constant-envelope modulation, deviation, sidebands, and phase-derivative demodulation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

from scipy.special import jv


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## FM Moves Information into Instantaneous Frequency

$$s_{FM}(t) = \cos\left(2\pi f_c t + 2\pi k_f \int_0^t m(\tau) d\tau\right)$$

The envelope stays constant. What changes is the density of zero crossings and the instantaneous phase slope.

In [ ]:
carrier_freq = 20_000
message = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
message = normalize(message)
freq_dev = 2_500
fm_signal = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=freq_dev)
fm_demod = fm_demodulate(fm_signal, fs=WORK_FS)

analytic = signal.hilbert(fm_signal)
inst_freq = np.diff(np.unwrap(np.angle(analytic))) * WORK_FS / (2 * np.pi)
inst_freq = np.append(inst_freq, inst_freq[-1])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
win = slice(0, 8000)
axes[0].plot(t_work[win] * 1000, fm_signal[win], color="tab:orange", linewidth=0.6)
axes[0].set_title("FM Waveform")
axes[0].set_xlabel("Time (ms)")
axes[1].plot(t_work[win] * 1000, inst_freq[win] - np.mean(inst_freq), color="tab:red", linewidth=0.8)
axes[1].set_title("Instantaneous Frequency")
axes[1].set_xlabel("Time (ms)")
plot_spectrum(fm_signal, fs=WORK_FS, ax=axes[2], title="FM Spectrum")
axes[2].set_xlim(0, 30_000)
axes[2].set_ylim(-100, 5)
plt.tight_layout()

display(Markdown("**FM demodulated audio**"))
display(audio_player(resample_signal(fm_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))


In [ ]:
betas = np.linspace(0, 5, 300)
orders = range(5)
fig, ax = plt.subplots(figsize=(10, 3.5))
for order in orders:
    ax.plot(betas, jv(order, betas), label=f"J{order}")
ax.set_title("Bessel Functions and FM Sideband Strength")
ax.set_xlabel("Modulation index beta")
ax.set_ylabel("Amplitude")
ax.legend(ncol=5)
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_fm(freq_dev=2500.0):
    fm_signal = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=freq_dev)
    demod = fm_demodulate(fm_signal, fs=WORK_FS)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(fm_signal[:8000], fs=WORK_FS, ax=axes[0], title=f"FM Waveform, dev={freq_dev:.0f} Hz")
    plot_spectrum(fm_signal, fs=WORK_FS, ax=axes[1], title="FM Spectrum")
    axes[1].set_xlim(0, 30_000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, resample_signal(demod, WORK_FS, PLAY_FS), rate=PLAY_FS)

controls = widgets.interactive(
    update_fm,
    freq_dev=float_slider(min_value=500, max_value=6000, step=100, value=2500, description="Dev Hz"),
)
display(controls, audio_out)


## Key Takeaway

FM hides the message in phase and frequency rather than amplitude. That makes it harder to understand at first glance, but it is also why FM handles amplitude noise so much better.